# The Ultimate AI Engineering Interview QA Guide

This notebook contains the comprehensive answers to all questions in the README.


## 1. Core Must-Know Concepts


**Q:** LLM Context: Standard pre-training vs. Instruction pre-training algorithms.
**Answer:** Standard pre-training predicts the next token in a vast corpus (e.g., CommonCrawl), creating a base model that understands language structure. Instruction pre-training (SFT) fine-tunes this base model on pairs of (instruction, output) to respond nicely to human queries.


**Q:** RAG Architecture: The baseline components.
**Answer:** Embed (convert docs to vectors) -> Retrieve (fetch relevant vectors using cosine similarity against the query vector) -> Generate (pass retrieved docs to LLM context to formulate an answer).


**Q:** Tool Use & Function Calling.
**Answer:** Allows an LLM to select from predefined tools, outputting a highly structured JSON. The deterministic backend executes the JSON args and passes the results back to the LLM.


**Q:** MCP (Model Context Protocol).
**Answer:** An open standard by Anthropic connecting foundation models securely to external enterprise data sources using a unified API, eliminating custom integrations.


**Q:** Agentic Workflows.
**Answer:** Uses loops, self-reflection, planning, and multi-agent roles to autonomously solve complex goals iteratively instead of single stateless prompts.


**Q:** PEFT / LoRA.
**Answer:** Parameter-Efficient Fine-Tuning. Instead of updating billions of parameters, LoRA freezes the original weights and inserts small, low-rank matrices into attention layers, drastically reducing compute memory needs.


**Q:** Quantization: Moving from FP16 to INT8/INT4.
**Answer:** Compresses weight precision to fit massive models on consumer GPUs, at the cost of highly marginal degradation in accuracy.


## 2. LLM Fundamentals & Architecture


**Q:** What are foundation models vs task-specific models?
**Answer:** Foundation models are massive, general-purpose models trained on terabytes of unlabelled data (zero-shot capable). Task-specific models (like a basic sentiment classifier) are purely tuned for one narrow task.


**Q:** Break down the Transformer architecture.
**Answer:** Encoders convert input sequences into contextual embeddings. Decoders predict the next token auto-regressively based on the encoder's output. Modern LLMs (GPT/Llama) are predominantly decoder-only.


**Q:** Explain tokenization and its impact.
**Answer:** Tokenizers (like BPE) group frequent characters into chunks. Poor tokenization splits numbers unpredictably, making arithmetic failure common in older LLMs. Modern models give individual numbers distinct tokens.


**Q:** What are positional embeddings? (Absolute, Relative, RoPE).
**Answer:** Transformers process tokens in parallel, lacking inherent order. Positional embeddings inject sequence info. Absolute embeddings add fixed values. RoPE (Rotary Position Embeddings) encodes relative distances geometrically by rotating vectors, allowing robust length extrapolation.


**Q:** Explain the Q, K, and V matrices.
**Answer:** Each token produces a Query (what I'm looking for), a Key (what I contain), and a Value (my actual content). Attention scores are calculated via dot product of Q and K, determining how much of V to pass forward.


**Q:** Why is the attention dot product scaled by $\sqrt{d_k}$?
**Answer:** To prevent dot products from growing excessively large, which would push softmax into flat regions with near-zero gradients (vanishing gradients).


**Q:** Explain causal masking.
**Answer:** A triangular mask of negative infinity applied to the attention matrix in decoders, preventing tokens from attending to "future" tokens they are trying to predict.


**Q:** MHA vs. GQA vs. MQA.
**Answer:** Multi-Head Attention (MHA) has unique Q, K, V for every head. Multi-Query Attention (MQA) shares one K and V head across all Q heads, drastically cutting memory. Grouped-Query Attention (GQA) is a middle ground, clustering heads to balance quality and speed.


**Q:** What is the KV Cache and PagedAttention?
**Answer:** KV Cache caches previous token keys and values so they aren't recalculated per step. PagedAttention fragments this cache into blocks logically connected by a page table (like OS vRAM), virtually eliminating memory fragmentation and boosting batch throughput.


**Q:** Logits, temperature, top-k, and top-p.
**Answer:** Logits are raw unnormalized scores. Temperature divides logits before softmax (T>1 increases randomness). Top-K strips all but K highest tokens. Top-P strictly samples from the highest probability tokens sequentially until weight sum P is reached.


**Q:** Mixture of Experts (MoE).
**Answer:** A sparse architecture consisting of multiple specialized "expert" neural networks per layer. A routing network sends a specific token only to the top 1 or 2 relevant experts, vastly increasing parameters without drastically increasing active compute cost during inference.


**Q:** Scenario: Penalize LLM repeating phrases.
**Answer:** Increase `repetition_penalty` or `presence_penalty` in the API, which subtracts a value from the logits of tokens that have already appeared frequently in the context window.


## 3. Prompt Engineering & Optimization


**Q:** Zero-shot, one-shot, and few-shot prompting.
**Answer:** Zero-shot provides no examples. One-shot provides one. Few-shot provides multiple context examples. Few-shot fails when examples introduce accidental statistical biases or when the reasoning is too complex to simply "pattern-match".


**Q:** CoT vs ToT.
**Answer:** Chain-of-Thought requires step-by-step sequential reasoning. Tree-of-Thought generates multiple branches of reasoning, evaluates each path's viability, and only proceeds down the most promising node (like a chess engine).


**Q:** ReAct (Reasoning + Acting) prompting.
**Answer:** Synthesizes CoT and action selection. The prompt asks the model to output a `Thought`, followed by an `Action` (tool call), returning the `Observation`. Great for agent loops.


**Q:** Engineering JSON structures.
**Answer:** Provide explicit dummy JSON schemas, enforce it with API parameters (e.g., OpenAI's `response_format: {type: "json_object"}`), or employ grammar-based sampling constraints in open source inference servers.


**Q:** Lost in the Middle phenomenon.
**Answer:** LLMs heavily bias attending to data at the very start or strictly the end of a long prompt. Mitigation: sort contexts by relevance, inject the most critical context immediately closest to the question at the bottom.


**Q:** Prompt Caching.
**Answer:** Caching the parsed KV representations of commonly used long prefixes (like a complex 10-page system prompt) into high-speed memory, saving compute latency and API costs on subsequent identical prefix queries.


**Q:** Scenario: Hitting context limits.
**Answer:** Utilize dynamic truncation, summarize older conversation context into rolling summaries, or use RAG to convert massive context histories into searchable external embeddings.


**Q:** Scenario: Overfitting on few-shot formats.
**Answer:** Provide a vastly heterogeneous set of few-shot examples (vary the lengths, structures, and tones of the desired answers) so the model extrapolates the logic rather than purely mimicking arbitrary stylistic patterns.


## 4. Retrieval-Augmented Generation (RAG)


**Q:** Architecture of chunk-and-embed RAG.
**Answer:** Parse Docs -> Split into chunks -> Embed via Model -> Store in Vector DB. Runtime: User Query -> Embed Query -> Retrieve Top Vector Cosine matches -> Pass as context string into generation LLM.


**Q:** Text chunking strategies.
**Answer:** Fixed-size (character count + overlap), Recursive (paragraph -> sentence -> word), Semantic (detect topic changes algorithmically before splitting), Parent-Child (search small sub-chunks but return the entire parent paragraph).


**Q:** Hybrid Search & RRF.
**Answer:** Keyword match (BM25) handles stark entity/serial numbers. Semantic Vector search handles concepts. RRF (Reciprocal Rank Fusion) recalculates the ranking dynamically by combining both algorithms' returned score lists into a unified relevance list.


**Q:** Re-ranking.
**Answer:** Standard bi-encoder retrieval lacks precision. Once Top 100 docs are fetched, a Cross-Encoder (Reranker) analyzes the Query and Document *together* sequentially to re-score and select the Top 5 most exquisitely relevant docs to inject into the LLM context.


**Q:** Query Transformation.
**Answer:** Re-writing the user query to retrieve better data. E.g., `Multi-Query` generates 3 variations of the same query. `Step-Back` abstracts the query to a higher level. `HyDE` creates a hypothetical dummy answer and searches for vectors similar to the dummy answer.


**Q:** Self-RAG / CRAG.
**Answer:** The LLM actively decides if it needs to search. If it does, it pulls context and scores the context relevance itself. If bad, it rewrites the query and tries again before giving up or answering, lowering hallucination rates dramatically.


**Q:** Graph RAG.
**Answer:** Parses raw text into Nodes (entities) and Edges (relationships) in a Knowledge Graph rather than dense vectors. Superior for complex, multi-hop reasoning (e.g., "Who is the director of the company that owns the server that crashed?").


**Q:** Scenario: Contradictory answers.
**Answer:** Attach robust metadata (creation dates, source tags). Before generation, sort and prioritize context blocks based on date, or explicitly prompt the LLM: "If information conflicts across contexts, trust the context with the most recent `Date` tag".


**Q:** Scenario: Slow RAG retrieval over 1M+ embeddings.
**Answer:** You must use HNSW (Hierarchical Navigable Small World) indices or IVFFlat clustering rather than exhaustive K-Nearest Neighbors (KNN), reducing search complexity from linear to logarithmic time.


## 5. AI Agents & Agentic Systems


**Answer Summaries:** Agents possess stateful memory loops and use tools. Plan-and-Execute agents break a task into a flowchart list, executing one step at a time via workers. DSPy uses compilation mathematically to optimize prompt weightings akin to PyTorch. Infinite loops are handled programmatically by enforcing hard `max_iteration` limits and propagating specific `Tracebacks` back to the Agent. Destructive actions require Human-in-the-Loop strict manual overrides before passing `--yes` to the execution shell.


## 6. Vector Databases & Embeddings


**Answer Summaries:** Dense models represent semantic meaning. Sparse models represent important word frequencies. HNSW is standard DB indexing. PQ compresses dimensions heavily. DB swapping requires dual-writing ingress events and running a massive backfilling script into the new embedding dimensions before cutting over API endpoints.


## 7. Fine-Tuning & Model Alignment


**Answer Summaries:** PEFT creates delta weights. SFT teaches form. RLHF aligns values using an external grading model (Reward Model). DPO removes the Reward Model entirely by comparing the log-probabilities of a "chosen" vs "rejected" completion directly in the LLM's loss function. Catastrophic forgetting is mitigated by interleaving the fine-tune dataset with portions of the original pre-training dataset as a grounding anchor.


## 8. AI System Design


**Answer Summaries:** Scaled Coding Assistant: Requires Fill-in-the-Middle (FIM) training and highly threaded, multi-level caching on device. Async Batch processing: Spin up Serverless Workers running vLLM that pull from a Pub/Sub queue aggressively at night. Routers check prompt complexity using a very cheap embedding/classification layer to forward to Claude 3.5 Sonnet vs Haiku, saving 10x API costs.


## 9. Production AI & LLMOps


**Answer Summaries:** LLMOps deals closely with stochastic outputs rather than static ML regressions. Observability requires tracing token counts to calculate hard monetary costs, and TTFT (Time To First Token) for latency tracking. Continuous batching processes strings stream-wise rather than locking up GPUs. Model Drift requires automated evaluation scripts continuously hitting models looking for benchmark degradation.


## 10. Evaluation & Benchmarking


**Answer Summaries:** BLEU is garbage for GenAI since synonyms are marked wrong. `LLM-as-a-judge` uses GPT-4 to grade outputs on a 1-5 rubric. RAGAS parses faithfulness (did the answer come exclusively from context?) and relevance (did it answer the query?). ELO ratings represent crowdsourced A/B blind tests. Resolving human vs LLM disagreements requires manual ground truth audits and adding explicitly defined negative constraints to the Judge's system prompt.


## 11. AI Infrastructure & Scalability


**Answer Summaries:** VRAM sets hard limits. Model weights + KV Cache + Activations must fit in memory. Tensor Parallelism splits individual weight matrices across multiple GPUs synchronously to run 70B parameters. Speculative decoding runs a tiny "draft" model to race ahead predicting tokens while the main model verifies them in chunks. Weights are offloaded to slow PCIe system RAM via libraries like DeepSpeed ZeRO-3 when OOM arrays are hit.


## 12. Multi-Modal AI


**Answer Summaries:** VLMs patch images into 16x16 squares, embed them linearly via a ViT, and concatenate them with text token embeddings. CLIP contrastively aligns images and text into the exact same vector space. Diffusion destroys images into static Gaussian noise, and the U-Net learns to reverse the noise conditioned on text embeddings. Video RAG pulls video transcripts, segments scenes chronologically, and embeds Keyframes using CLIP bindings.


## 13. AI Safety, Ethics & Security


**Answer Summaries:** Prompt injection forces the model to ignore prior commands (Indirect injection arrives securely via poisoned retrieved documents). Defenses use randomized delimiter fences (`<USER_INPUT_{RANDOM_HASH}>`). Data poisoning maliciously alters pre-training sets. Differential privacy injects mathematical noise into gradients to prevent the LLM from perfectly memorizing/leaking exact passwords or PII. High Risk EU systems require transparent external bias auditing logs.


## 14. Coding & Practical Implementation


**Answer Summaries:** Implementations require heavy proficiency with NumPy for fast vector dot products representing semantic caches. Asyncio is heavily leveraged to launch 10 API calls simultaneously rather than sequentially. Chunkers utilize RecursiveCharacter frameworks splitting recursively from `\n\n` down to ` `.


## 15. Behavioral & Scenario-Based Questions


**Answer Summaries:** Root causes of hallucinations are tracked via tracing libraries (LangSmith); usually, the underlying Top-K retrieval fetched the wrong chunk. RAG > Fine Tuning is justified to management directly via ROI metrics: fine tuning cannot "learn" factually dense, ever-changing databases dynamically and is incredibly expensive to train frequently. High budget constraints demand local hosting using quantized (GGUF) models served on cheap edge devices, drastically reducing per-token external API taxes.
